In [1]:
import sys
!{sys.executable} -m pip install -U pip
!{sys.executable} -m pip install matplotlib pillow tqdm
!{sys.executable} -m pip install "monai-weekly"

In [2]:
import sys, subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "matplotlib", "pillow", "tqdm", "scikit-learn", "tensorboard", "monai-weekly"])
print("Install OK ✅")

Install OK ✅


In [3]:
import sys
!{sys.executable} -m pip install -U scikit-learn

In [4]:
import os
import shutil
import tempfile
import matplotlib.pyplot as plt
import PIL
import torch
from torch.utils.tensorboard import SummaryWriter
import numpy as np
from sklearn.metrics import classification_report

from monai.apps import download_and_extract
from monai.config import print_config
from monai.data import decollate_batch, DataLoader
from monai.metrics import ROCAUCMetric
from monai.networks.nets import DenseNet121
from monai.transforms import (
    Activations,
    EnsureChannelFirst,
    AsDiscrete,
    Compose,
    LoadImage,
    RandFlip,
    RandRotate,
    RandZoom,
    ScaleIntensity,
)
from monai.utils import set_determinism

print_config()

MONAI version: 1.6.dev2609
Numpy version: 2.4.2
Pytorch version: 2.10.0
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: 2bbc952e6f4a983e3c84e697a01b49f4344b4552
MONAI __file__: /opt/homebrew/lib/python3.11/site-packages/monai/__init__.py

Optional dependencies:
Pytorch Ignite version: NOT INSTALLED or UNKNOWN VERSION.
ITK version: NOT INSTALLED or UNKNOWN VERSION.
Nibabel version: NOT INSTALLED or UNKNOWN VERSION.
scikit-image version: NOT INSTALLED or UNKNOWN VERSION.
scipy version: 1.17.1
Pillow version: 12.1.1
Tensorboard version: 2.20.0
gdown version: NOT INSTALLED or UNKNOWN VERSION.
TorchVision version: NOT INSTALLED or UNKNOWN VERSION.
tqdm version: 4.67.3
lmdb version: NOT INSTALLED or UNKNOWN VERSION.
psutil version: 6.0.0
pandas version: NOT INSTALLED or UNKNOWN VERSION.
einops version: NOT INSTALLED or UNKNOWN VERSION.
transformers version: NOT INSTALLED or UNKNOWN VERSION.
mlflow version: NOT INSTALLED or UNKNOWN VERSION.
pynrrd versi

In [5]:
import os
from pathlib import Path

# Repo-root: hvis notebooken ligger i src/ og du kjører den derfra
repo_root = Path.cwd().resolve().parent  # src -> CancerVision

# Datasettet er allerede pakket ut her:
data_dir = repo_root / "res" / "archive" / "NINS_Dataset" / "NINS_Dataset"

if not data_dir.exists():
    raise FileNotFoundError(f"Fant ikke {data_dir}. Sjekk at datasettet ligger i res/archive")

print("Bruker datasett fra:", data_dir.resolve())
print("Toppnivå-filer/mapper:", [p.name for p in data_dir.iterdir()][:30])

Bruker datasett fra: /Users/tobiasamundrud/Documents/Cogito/CancerVision/res/archive/NINS_Dataset/NINS_Dataset
Toppnivå-filer/mapper: ['Leukoencephalopathy with subcortical cysts', 'Encephalomalacia with gliotic change', 'Ischemic change  demyelinating plaque', 'Malformation (Chiari I)', 'Cerebral Hemorrhage', 'Brain tumor (Dermoid cyst craniopharyngioma)', 'Obstructive Hydrocephalus', 'Postoperative encephalomalacia', 'Brain tumor (Astrocytoma Ganglioglioma)', 'Brain Atrophy', 'Brain tumor - Recurrenceremnant of previous lesion', 'Post-operative Status with Small Hemorrhage', 'Brain Infection', 'Stroke (Haemorrhage)', 'Glioma', 'Brain Infection with abscess', 'models', 'Brain tumor operated with ventricular hemorrhage', 'cerebral venous sinus thrombosis', 'Microvascular ischemic change', 'Small Vessel Diease Demyelination', 'Mid triventricular hydrocephalus', 'Stroke (Demyelination)', 'Brain Tumor (Ependymoma)', 'White Matter Disease', 'Cerebral abscess', 'NMOSD  ADEM', 'Brain Tumor (

In [6]:
from pathlib import Path

data_dir = Path(data_dir)
print("data_dir:", data_dir.resolve())

# Vis toppnivå
top = [p for p in data_dir.iterdir()]
print("Toppnivå:", [p.name for p in top])

# Hvis toppnivå er mapper: sjekk om de ser ut som klasser
class_dirs = [p for p in top if p.is_dir()]
if class_dirs:
    for d in class_dirs[:10]:
        n = len(list(d.rglob("*.png"))) + len(list(d.rglob("*.jpg"))) + len(list(d.rglob("*.jpeg")))
        print(f"- {d.name}: {n} bilder (png/jpg/jpeg)")
else:
    print("Ingen klasse-mapper på toppnivå (kan være annen struktur).")


data_dir: /Users/tobiasamundrud/Documents/Cogito/CancerVision/res/archive/NINS_Dataset/NINS_Dataset
Toppnivå: ['Leukoencephalopathy with subcortical cysts', 'Encephalomalacia with gliotic change', 'Ischemic change  demyelinating plaque', 'Malformation (Chiari I)', 'Cerebral Hemorrhage', 'Brain tumor (Dermoid cyst craniopharyngioma)', 'Obstructive Hydrocephalus', 'Postoperative encephalomalacia', 'Brain tumor (Astrocytoma Ganglioglioma)', 'Brain Atrophy', 'Brain tumor - Recurrenceremnant of previous lesion', 'Post-operative Status with Small Hemorrhage', 'Brain Infection', 'Stroke (Haemorrhage)', 'Glioma', 'Brain Infection with abscess', 'models', 'Brain tumor operated with ventricular hemorrhage', 'cerebral venous sinus thrombosis', 'Microvascular ischemic change', 'Small Vessel Diease Demyelination', 'Mid triventricular hydrocephalus', 'Stroke (Demyelination)', 'Brain Tumor (Ependymoma)', 'White Matter Disease', 'Cerebral abscess', 'NMOSD  ADEM', 'Brain Tumor (Hemangioblastoma  Pleomo

## Lager class_names, files og labels 

In [7]:
import numpy as np
from pathlib import Path

data_dir = Path(data_dir) #gjør om data_dir til et Path-objekt for enklere filhåndtering

IMG_EXTS = [".png", ".jpg", ".jpeg"] #kun bruke disse filtypene

class_names = sorted([d.name for d in data_dir.iterdir() if d.is_dir()]) # iterdir lister alt i toppnivåt, is_dir tar bare med mapper, name tar bare med navnet på mappa og ikke full sti, sorterer alfabetisk
class_to_idx = {c: i for i, c in enumerate(class_names)} #lager dictionary med navn og tall-label som peker på hverandre 

files, labels = [], []
for c in class_names: #går gjennom hver klasse-mappe
    for p in (data_dir / c).rglob("*"): #lagrer stien til mappa og finner alle filer og mapper inni
        if p.suffix.lower() in IMG_EXTS: #sjekker om fila ender på en av de godkjente filtypene
            files.append(str(p)) #legger inn full filsti som string i files-lista
            labels.append(class_to_idx[c]) #legger inn label (tall) som tilsvarer klassenavnet i labels-lista

print("Antall klasser:", len(class_names))
print("Klasser (første 15):", class_names[:15])
print("Antall bilder totalt:", len(files))


Antall klasser: 38
Klasser (første 15): ['Brain Atrophy', 'Brain Infection', 'Brain Infection with abscess', 'Brain Tumor', 'Brain Tumor (Ependymoma)', 'Brain Tumor (Hemangioblastoma  Pleomorphic xanthroastrocytoma  metastasis)', 'Brain tumor (Astrocytoma Ganglioglioma)', 'Brain tumor (Dermoid cyst craniopharyngioma)', 'Brain tumor - Recurrenceremnant of previous lesion', 'Brain tumor operated with ventricular hemorrhage', 'Cerebral Hemorrhage', 'Cerebral abscess', 'Encephalomalacia with gliotic change', 'Glioma', 'Hemorrhagic collection']
Antall bilder totalt: 5250


## Stratified split

In [8]:
from sklearn.model_selection import train_test_split

train_files, test_files, train_labels, val_labels = train_test_split(files, labels, test_size=0.2, random_state=0, stratify=labels) #Deler inn i train og val sett, 

train_data = [{"image": f, "label": l} for f, l in zip(train_files, train_labels)]
val_data = [{"image": f, "label": l} for f, l in zip(test_files, val_labels)] #kobler sammen filsti med label

print("Train:", len(train_data), "Val:", len(val_data)) #


Train: 4200 Val: 1050


## Transforms + loaders (2D klassifikasjon)

In [9]:
import torch
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, ScaleIntensityd,
    ResizeWithPadOrCropd, RandFlipd, RandRotate90d, EnsureTyped
)
from monai.data import Dataset, DataLoader

train_tfms = Compose([ #transformasjoner som skal brukes på treningsdata
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    ScaleIntensityd(keys=["image"]),
    ResizeWithPadOrCropd(keys=["image"], spatial_size=(224, 224)),
    RandFlipd(keys=["image"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image"], prob=0.5, spatial_axis=1),
    RandRotate90d(keys=["image"], prob=0.5, max_k=3),
    EnsureTyped(keys=["image", "label"]),
])

val_tfms = Compose([ #transformasjoner som skal brukes på valideringsdata (ingen augmentering)
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    ScaleIntensityd(keys=["image"]),
    ResizeWithPadOrCropd(keys=["image"], spatial_size=(224, 224)),
    EnsureTyped(keys=["image", "label"]),
])

train_ds =Dataset(train_data, transform=train_tfms) #lager dataset-objekter som kobler sammen data og transformasjoner
val_ds = Dataset(val_data, transform=val_tfms)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2) #lager dataloadere som håndterer batching og shuffling av data under trening
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

device = torch.device(
            "cuda"
            if torch.cuda.is_available()
            else "mps" if torch.backends.mps.is_available() else "cpu"
        ) #sjekker om det finnes en GPU som kan brukes, hvis ikke bruker den CPU
print("Device:", device)

Device: mps


## Modell + training loop

In [10]:
from monai.networks.nets import DenseNet121

num_classes = len(class_names) #antall mapper

model = DenseNet121(    #dense er selve AI-modellen som skal trenes, 121 er en spesifikk variant av DenseNet-arkitekturen
    spatial_dims=2, #inputbilder er 2D
    in_channels=3,          #bildene er enkanals (1 kanal) 
    out_channels=num_classes    #modellen gir en skår for hver klasse
).to(device) #flytter AI-modellen til GPU hvis tilgjengelig, ellers CPU

loss_fn = torch.nn.CrossEntropyLoss() #måler hvor feil modellen tar
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3) #oppdaterer modellens vekter så loss blir lavere neste gang, lr=1e-3 er hvor store steg den tar når der lærer

def acc(logits, y): #logits er modellens score for hver klasse, y er de riktige labelene
    return (logits.argmax(dim=1) == y).float().mean().item() #velger klassen med høyest score

max_epochs = 5 #5 runder gjennom datasettet
for epoch in range(max_epochs): 
    model.train()  
    running_loss, running_acc = 0.0, 0.0

    for batch in train_loader:
        x = batch["image"].to(device)
        y = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(x)            #1. modellen henter en batch bilder + labels 2. modellen gjetter 3. regner hvor feil 4. lærer av feilen
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()     #legger sammen loss og accuracy for alle batches, deler på antall batches for å få gjennomsnitt per epoch
        running_acc += acc(logits.detach(), y)

    train_loss = running_loss / max(1, len(train_loader))
    train_acc  = running_acc / max(1, len(train_loader))

    model.eval()    #setter modellen i test-modus
    val_acc = 0.0  
    with torch.no_grad():  #lærer ikke, bare evaluerer
        for batch in val_loader:
            x = batch["image"].to(device) 
            y = batch["label"].to(device)
            logits = model(x)
            val_acc += acc(logits, y)
    val_acc /= max(1, len(val_loader)) #sier hvor bra den gjør det på nye bilder 

    print(f"Epoch {epoch+1}/{max_epochs} | loss={train_loss:.4f} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f}")

KeyboardInterrupt: 